In [20]:
#Import the libraies 
import pandas as pd, numpy as np, json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [22]:
#Read the files
Menu_Clean=pd.read_csv("mcdonalds_menu_clean.csv")
Breakfast_Clean=pd.read_csv("breakfast_basket_clean.csv")
US_income=pd.read_csv("US income 2026 - Sheet1.csv")
#The data files loaded here will be consistently used in the data for the analysis in this file

In [23]:
# ---------- 1. Descriptive stats ----------
print("MENU price_usd: mean %.2f median %.2f mode %.2f"%(Menu_Clean.price_usd.mean(),Menu_Clean.price_usd.median(),Menu_Clean.price_usd.mode()[0]))
print(Menu_Clean.groupby("category").price_usd.agg(["mean","median","count"]).round(2))
bb=Breakfast_Clean.drop_duplicates(["City","Month"])
print("BASKET USD: mean %.2f median %.2f mode %.2f"%(bb.Breakfast_Basket_USD.mean(),bb.Breakfast_Basket_USD.median(),bb.Breakfast_Basket_USD.mode()[0]))
print(Breakfast_Clean.groupby("Item_Category").Price_USD.agg(["mean","median"]).round(2))

MENU price_usd: mean 4.08 median 3.98 mode 1.67
                      mean  median  count
category                                 
Beverages             2.38    2.45     12
Breakfast             5.14    5.03     10
Burgers               5.88    5.98     10
Chicken & Sandwiches  5.52    4.98      7
Desserts              2.98    2.87     11
Happy Meal            5.69    5.62      4
McCafé                2.93    2.92     12
Salads                5.76    5.82      5
Snacks & Sides        3.32    3.53      9
BASKET USD: mean 9.56 median 9.16 mode 10.65
                mean  median
Item_Category               
Bakery          1.80    1.65
Dairy & Eggs    5.41    3.03
Fruits          2.48    2.19
Grains          3.09    2.48
Meat           12.72   10.86
Vegetables      2.18    1.72


In [24]:
# ---------- 2. Time trend: menu price history vs basket ----------
rows=[]
for _,r in Menu_Clean.iterrows():
    for h in json.loads(r.price_history): rows.append({"item":r.item_name,"category":r.category,**h})
hist=pd.DataFrame(rows)
menu_year=hist.groupby("year").price_usd.mean()
print("\nAvg menu price by year:\n",menu_year.round(3))
us=Breakfast_Clean[Breakfast_Clean.Country=="United States"].drop_duplicates(["City","Month"])
us_month=us.groupby("Month").Breakfast_Basket_USD.mean()
glob_month=bb.groupby("Month").Breakfast_Basket_USD.mean()
print("\nUS basket by month:\n",us_month.round(2))
print("Basket change Oct25->Mar26: global %.1f%%, US %.1f%%"%((glob_month.iloc[-1]/glob_month.iloc[0]-1)*100,(us_month.iloc[-1]/us_month.iloc[0]-1)*100))
print("Menu change 2025->2026: %.1f%%"%((menu_year[2026]/menu_year[2025]-1)*100))


Avg menu price by year:
 year
2023    4.376
2024    4.286
2025    4.208
2026    4.084
Name: price_usd, dtype: float64

US basket by month:
 Month
2025-10    15.02
2025-11    15.06
2025-12    15.45
2026-01    15.29
2026-02    15.50
2026-03    15.50
Name: Breakfast_Basket_USD, dtype: float64
Basket change Oct25->Mar26: global 3.7%, US 3.1%
Menu change 2025->2026: -2.9%


In [25]:
# ---------- 3. Correlation ----------
fao=Breakfast_Clean.groupby("Month")[["FAO_Index_Value","Breakfast_Basket_USD"]].mean()
print("\nCorr FAO index vs basket (monthly):",round(fao.corr().iloc[0,1],3))
num=Menu_Clean.select_dtypes("number")
print("\nMenu price correlations:\n",num.corr()["price_usd"].sort_values(ascending=False).round(2).head(8))
# category-level: share of dairy/egg/bread basket items vs breakfast menu price
cat_basket=Breakfast_Clean.groupby("Item_Category").Price_USD.mean()
print("\nBreakfast menu mean price: %.2f"%Menu_Clean[Menu_Clean.category=="Breakfast"].price_usd.mean())



Corr FAO index vs basket (monthly): -0.445

Menu price correlations:
 price_usd            1.00
sodium_mg            0.67
protein_g            0.59
trans_fat_g          0.57
total_fat_g          0.54
saturated_fat_g      0.49
calories             0.41
calories_from_fat    0.41
Name: price_usd, dtype: float64

Breakfast menu mean price: 5.14


In [26]:
# ---------- 4. Machine learning (what drives menu price) ----------
feats=["calories","total_fat_g","protein_g","sodium_mg","total_carbs_g","sugars_g","prep_time_minutes"]
X=Menu_Clean[feats].fillna(0); y=Menu_Clean.price_usd
X=pd.concat([X,pd.get_dummies(Menu_Clean.category,dtype=float)],axis=1)
print("\nLinearReg CV R2: %.2f"%cross_val_score(LinearRegression(),X,y,cv=5,scoring="r2").mean())
rf=RandomForestRegressor(300,random_state=0)
print("RandomForest CV R2: %.2f"%cross_val_score(rf,X,y,cv=5,scoring="r2").mean())
rf.fit(X,y); print(pd.Series(rf.feature_importances_,X.columns).sort_values(ascending=False).head(6).round(3))
 
# 2026 scenario: apply basket inflation to breakfast-menu items
infl=(us_month.iloc[-1]/us_month.iloc[0]-1)


LinearReg CV R2: -0.91
RandomForest CV R2: -0.08
sodium_mg      0.583
protein_g      0.076
sugars_g       0.076
calories       0.049
total_fat_g    0.043
Salads         0.037
dtype: float64


In [27]:
# ---------- 4b. Feature selection ----------
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
 
all_feats=[c for c in num.columns if c not in ["price_usd","price_per_1000_cal","protein_per_dollar"]]  # drop price-derived leakage
Xf=Menu_Clean[all_feats].fillna(0); Xf=Xf.loc[:,Xf.std()>0]
 
# (a) Univariate: F-test
skb=SelectKBest(f_regression,k=5).fit(Xf,y)
print("\nSelectKBest top 5:",list(Xf.columns[skb.get_support()]))
 
# (b) RFE with random forest
rfe=RFE(RandomForestRegressor(200,random_state=0),n_features_to_select=5).fit(Xf,y)
print("RFE top 5:",list(Xf.columns[rfe.support_]))
 
# (c) Lasso (keeps features with non-zero coefficients)
lasso=make_pipeline(StandardScaler(),LassoCV(cv=5,random_state=0)).fit(Xf,y)
coef=pd.Series(lasso[-1].coef_,Xf.columns)
print("Lasso kept:",list(coef[coef!=0].abs().sort_values(ascending=False).index))
 
# (d) Does selection help? Compare CV R2
sets={"All numeric":list(Xf.columns),
      "SelectKBest":list(Xf.columns[skb.get_support()]),
      "RFE":list(Xf.columns[rfe.support_]),
      "Lasso":list(coef[coef!=0].index)}
for name,cols in sets.items():
    if cols:
        r2=cross_val_score(RandomForestRegressor(200,random_state=0),Xf[cols],y,cv=5,scoring="r2").mean()
        print("RF CV R2 with %-12s (%2d feats): %.2f"%(name,len(cols),r2))
 
slope=np.polyfit(range(len(us_month)),us_month.values,1)[0]
print("\nUS basket trend: %+.3f USD/month"%slope)
 


SelectKBest top 5: ['total_fat_g', 'saturated_fat_g', 'trans_fat_g', 'sodium_mg', 'protein_g']
RFE top 5: ['cholesterol_mg', 'sodium_mg', 'sugars_g', 'protein_g', 'vitamin_a_iu']
Lasso kept: ['sodium_mg', 'trans_fat_g', 'total_carbs_g', 'vitamin_a_iu']
RF CV R2 with All numeric  (16 feats): -0.05
RF CV R2 with SelectKBest  ( 5 feats): -0.12
RF CV R2 with RFE          ( 5 feats): 0.03
RF CV R2 with Lasso        ( 4 feats): -0.05

US basket trend: +0.101 USD/month


In [28]:
# ---------- Plots ----------
fig,ax=plt.subplots(2,2,figsize=(13,9))
Menu_Clean.groupby("category").price_usd.mean().sort_values().plot.barh(ax=ax[0,0],title="Avg menu price by category (2026)")
menu_year.plot(marker="o",ax=ax[0,1],title="Avg menu price by year");ax[0,1].set_ylabel("USD")
ax[1,0].plot(glob_month.index,glob_month.values,marker="o",label="Global");ax[1,0].plot(us_month.index,us_month.values,marker="s",label="US");ax[1,0].set_title("Breakfast basket USD (Oct25-Mar26)");ax[1,0].legend()
ax[1,1].scatter(Menu_Clean.calories,Menu_Clean.price_usd,alpha=.6);ax[1,1].set_title("Menu: calories vs price");ax[1,1].set_xlabel("calories");ax[1,1].set_ylabel("USD")
plt.tight_layout();plt.savefig("analysis.png",dpi=120)

In [29]:
# 5. INCOME ANALYSIS (Census Table F-1: upper limit of each family-income quintile)
# --- 5a. Clean the messy Census sheet (two tables stacked: current $ and 2025 $) ---
cols=["year","families_k","Q1","Q2","Q3","Q4","Top5"]
US_income.columns=cols
i_cur =US_income.index[US_income.year.str.strip()=="Current Dollars"][0]
i_real=US_income.index[US_income.year.str.strip()=="2025 Dollars"][0]
def tidy(df):
    df=df[df.year.str.match(r"^\d{4}(\s*\(\d+\))?$",na=False)].copy()  # keep "2024" / "2024 (42)", drop "2025 Dollars" header
    df["year"]=df.year.str[:4].astype(int)                         # "2024 (42)" -> 2024
    for c in cols[1:]: df[c]=pd.to_numeric(df[c].str.replace(",",""),errors="coerce")
    return df.drop_duplicates("year").set_index("year").sort_index()  # keep first row where a year repeats
inc_nom =tidy(US_income.loc[i_cur:i_real-1])    # current dollars
inc_real=tidy(US_income.loc[i_real:])           # 2025 dollars
Q=["Q1","Q2","Q3","Q4"]
print("\n=== INCOME (upper limit of quintile, family income) ===")
print(inc_nom.loc[2019:2025,Q].astype(int))
 
# --- 5b. Income growth by quintile ---
def cagr(df,a,b): return ((df.loc[b,Q]/df.loc[a,Q])**(1/(b-a))-1)*100
print("\nNominal CAGR 2019-2025 (%):\n",cagr(inc_nom,2019,2025).round(2).to_dict())
print("Real (2025$) CAGR 2019-2025 (%):\n",cagr(inc_real,2019,2025).round(2).to_dict())
print("Nominal growth 2023->2025 (%):",((inc_nom.loc[2025,Q]/inc_nom.loc[2023,Q]-1)*100).round(1).to_dict())
 
# --- 5c. Menu affordability: avg menu item as % of weekly family income ---
avg_menu =hist.groupby("year").price_usd.mean()
avg_bkfst=hist[hist.category=="Breakfast"].groupby("year").price_usd.mean()
weekly=inc_nom[Q]/52
aff=pd.DataFrame({q:avg_menu.loc[2023:2025]/weekly.loc[2023:2025,q]*100 for q in Q})
print("\nAvg menu item as % of weekly family income (2023-2025):\n",aff.round(4))
print("\nBreakfast items: price vs income, 2023-2025:")
print(pd.DataFrame({"avg_breakfast_price":avg_bkfst.loc[2023:2025],"Q1_limit":inc_nom.loc[2023:2025,"Q1"]}).round(2))
 
# --- 5d. Correlation: income vs menu price (ONLY 3 overlapping years, so n=3: indicative only) ---
ov=pd.concat([avg_menu.loc[2023:2025].rename("menu_price"),inc_nom.loc[2023:2025,Q]],axis=1)
print("\nCorrelation of avg menu price with income limits (n=3 years!):\n",ov.corr()["menu_price"].drop("menu_price").round(3).to_dict())
# Longer-run check: income vs Food-at-home outlook is not available, so compare income growth vs menu change per item
chg=hist.pivot(index="item",columns="year",values="price_usd")
print("Items whose price FELL 2023->2026: %d of %d"%(((chg[2026]<chg[2023]).sum()),len(chg)))
 
# --- 5e. Grocery basket (milk 1L + bread 500g + eggs 12 + rice 1kg) vs income ---
# The Breakfast_Basket_USD column equals exactly the sum of these 4 items
us_basket=us_month.iloc[-1]; us_basket0=us_month.iloc[0]
share=pd.DataFrame({"Oct 2025":us_basket0/weekly.loc[2025,Q]*100,"Mar 2026":us_basket/weekly.loc[2025,Q]*100})
print("\nUS basket as % of WEEKLY family income (upper limit of each quintile, 2025 income):\n",share.round(2))
bk=Menu_Clean[Menu_Clean.category=="Breakfast"].price_usd.mean()
print("\nUS basket (Mar 2026) = %.2f USD = %.1f average McDonald's breakfast items (%.2f each)"%(us_basket,us_basket/bk,bk))
by_city=us[us.Month=="2026-03"].set_index("City").Breakfast_Basket_USD.sort_values()
print("\nUS basket by city, Mar 2026:\n",by_city.round(2).to_dict())
 
# --- 5f. Forecast 2026 income limits (backtested), then 2026 affordability ---
# A log-trend fitted from 2000 undershoots 2025 (inflation shock), so compare simple forecasters
# by rolling 1-step-ahead backtest (predict 2021..2025 using only earlier years).
from sklearn.metrics import mean_absolute_percentage_error as mape
def f_naive(d):  return d.iloc[-1]
def f_cagr5(d):  return d.iloc[-1]*(d.iloc[-1]/d.iloc[-6])**(1/5)
def f_lin10(d):
    w=d.iloc[-10:]; m=LinearRegression().fit(w.index.values.reshape(-1,1),w.values)
    return float(m.predict([[d.index[-1]+1]])[0])
methods={"naive (no growth)":f_naive,"trailing 5-yr CAGR":f_cagr5,"linear trend, last 10 yrs":f_lin10}
res={}
for name,f in methods.items():
    errs=[]
    for q in Q:
        d=inc_nom[q].loc[2000:2025]
        pred=[f(d.loc[:y-1]) for y in range(2021,2026)]
        errs.append(mape(d.loc[2021:2025].values,pred)*100)
    res[name]=np.mean(errs)
print("\nBacktest MAPE 2021-2025 (avg of quintiles):",{k:round(v,1) for k,v in res.items()})
best=min(res,key=res.get); print("Best method:",best)
inc26=pd.Series({q:methods[best](inc_nom[q].loc[2000:2025]) for q in Q})
print("2026 income-limit forecast:",inc26.round(0).astype(int).to_dict())
aff26=avg_menu[2026]/(inc26/52)*100
print("Avg 2026 menu item as %% of weekly family income: %s"%aff26.round(4).to_dict())
print("Relative change vs 2025 (%%): %s"%((aff26/aff.loc[2025]-1)*100).round(1).to_dict())
 
# --- 5g. Plots ---
fig,ax=plt.subplots(2,2,figsize=(13,9))
(inc_real.loc[2019:2025,Q]/inc_real.loc[2019,Q]*100).plot(marker="o",ax=ax[0,0],title="Real family income limits (2019=100)")
aff.plot(marker="o",ax=ax[0,1],title="Avg menu item as % of weekly family income"); ax[0,1].set_ylabel("%")
share.plot.bar(ax=ax[1,0],title="4-staple basket as % of weekly income (US)"); ax[1,0].set_ylabel("%"); ax[1,0].tick_params(axis="x",rotation=0)
by_city.plot.barh(ax=ax[1,1],title="US basket by city, Mar 2026 (USD)")
plt.tight_layout(); plt.savefig("analysis_income.png",dpi=120)


=== INCOME (upper limit of quintile, family income) ===
         Q1     Q2      Q3      Q4
year                              
2019  40000  69000  105000  164900
2020  38200  67760  103900  164500
2021  40000  70510  109400  174000
2022  42350  74500  114000  178000
2023  46000  80540  122700  194600
2024  47780  84160  130200  206500
2025  50230  88380  136500  213400

Nominal CAGR 2019-2025 (%):
 {'Q1': 3.87, 'Q2': 4.21, 'Q3': 4.47, 'Q4': 4.39}
Real (2025$) CAGR 2019-2025 (%):
 {'Q1': 0.19, 'Q2': 0.52, 'Q3': 0.76, 'Q4': 0.69}
Nominal growth 2023->2025 (%): {'Q1': 9.2, 'Q2': 9.7, 'Q3': 11.2, 'Q4': 9.7}

Avg menu item as % of weekly family income (2023-2025):
           Q1      Q2      Q3      Q4
year                                
2023  0.4946  0.2825  0.1854  0.1169
2024  0.4665  0.2648  0.1712  0.1079
2025  0.4356  0.2476  0.1603  0.1025

Breakfast items: price vs income, 2023-2025:
      avg_breakfast_price  Q1_limit
year                               
2023                 5.48   